# Lab 3.1, Build 2: Write a Strategy Router

Tina has three retrieval strategies: naive (semantic query), advanced (metadata filter), agentic (ReAct loop).
Your job: write the routing logic that picks the right one for each query.

In [ ]:
import os, json, pathlib, sys
from elasticsearch import Elasticsearch
from tina import strategy_router as default_router
from tina.guardrails import GUARDRAIL_HOOKS

es = Elasticsearch(os.environ["ES_ENDPOINT"], api_key=os.environ["ES_API_KEY"])

# Dev questions (12 labelled queries, public)
dev_queries = json.loads(pathlib.Path("/home/elastic/dev-sets/cases-dev-queries.json").read_text())
print(f"Loaded {len(dev_queries)} dev queries")

# Feature extractor (provided)
def extract_features(query: str) -> dict:
    q = query.lower()
    return {
        "has_filter_intent": any(w in q for w in ["structuring", "wire fraud", "sanctions", "kyc", "elder"]),
        "asks_for_figure": any(w in q for w in ["how much", "amount", "total", "$", "threshold"]),
        "multi_hop": any(w in q for w in ["both", "all", "compare", "across", "between"]),
        "has_case_id": any(c in q for c in ["cx-", "case #", "case no"]),
    }

In [ ]:
# YOUR WORK: Override strategy_router with your logic.
# features keys: has_filter_intent, asks_for_figure, multi_hop, has_case_id

def strategy_router(query: str, features: dict) -> str:
    """Return 'naive', 'advanced', or 'agentic'."""
    # Replace this with your routing logic:
    return "naive"  # placeholder — a constant fails the check (balanced set)

In [ ]:
correct = 0
for q in dev_queries:
    features = extract_features(q["text"])
    predicted = strategy_router(q["text"], features)
    label = q["retrieval_pattern"]
    match = "\u2713" if predicted == label else "\u2717"
    print(f"{match} [{label}\u2192{predicted}] {q['text'][:60]}")
    if predicted == label:
        correct += 1

print(f"\nAccuracy: {correct}/{len(dev_queries)} = {correct/len(dev_queries):.0%}")
print("Target: 0.80 on the held-out set (15 queries). Aim for 10/12 on dev before submitting.")

In [ ]:
import datetime

traces_dir = pathlib.Path("/home/elastic/.traces")
traces_dir.mkdir(exist_ok=True)

trace_record = {
    "challenge": "03-let-tina-choose-a-strategy",
    "written_at": datetime.datetime.utcnow().isoformat(),
    "dev_accuracy": correct / max(len(dev_queries), 1),
    "router_source": "strategy_router defined in notebook",
}
(traces_dir / "router-traces.json").write_text(json.dumps(trace_record, indent=2))
print("Trace saved. Select Check to submit.")